# HyperData ↔ Lumen Integration Tutorial

This notebook demonstrates Lumen's **scientific image framework** combined
with HyperData's **versioned dataset storage**.

| Section | Lumen Feature | Description |
|---------|---------------|-------------|
| §1 | `HyperDataImageDataset` / `HyperDataSegmentationDataset` | Pull & load remote datasets |
| §2 | `default_seg_aug` (YOLO-style pipeline) | Augment data with synced image+mask transforms |
| §3 | `EncoderBase` + `MAETrainer` + `train_self_supervised_epoch` | Self-supervised pretraining |
| §4 | `tokens_to_pca_rgb` | Visualize learned encoder features |
| §5 | `SegmentationTrainer` + `train_fine_tune_epoch` | Downstream segmentation fine-tuning |
| §6 | `mean_iou` / `dice_coefficient` / `pixel_accuracy` | Evaluate segmentation quality |
| §7 | `push_to_hub` + `WeightManager` | Push dataset & weights to HyperData |
| §8 | `WeightManager.pull_weights` | Pull weights and resume training |

## 0. Setup

```bash
uv pip install -e ".[hyperdata]"
```

The cell below sets default environment variables for the shared HyperData
deployment. Override them before running if your server differs.

In [ ]:
import os

# ---- Default HyperData remote configuration ----
os.environ.setdefault("HYPERDATA_ENDPOINT", "http://118.180.19.234:8021")
os.environ.setdefault("MINIO_ENDPOINT", "118.180.19.234")
os.environ.setdefault("MINIO_PORT", "9010")
os.environ.setdefault("MINIO_ACCESS_KEY", "hyperdata_admin")
os.environ.setdefault("MINIO_SECRET_KEY", "change-this-password-in-production")
os.environ.setdefault("S3_ENDPOINT", "118.180.19.234")
os.environ.setdefault("S3_PORT", "9010")
os.environ.setdefault("S3_ACCESS_KEY", "hyperdata_admin")
os.environ.setdefault("S3_SECRET_KEY", "change-this-password-in-production")
os.environ.setdefault("S3_BUCKET", "hyperdata-data")

import numpy as np
import torch
from hyperdata import HyperData

# ---- Lumen core imports ----
from lumen.models.encoder_base import EncoderBase
from lumen.models.heads import SegmentationHead
from lumen.models.feature_viz import tokens_to_pca_rgb
from lumen.training import MAETrainer, SegmentationTrainer
from lumen.training.workflow import train_self_supervised_epoch, train_fine_tune_epoch
from lumen.training.eval import mean_iou, dice_coefficient, pixel_accuracy
from lumen.data.augment import default_seg_aug
from lumen.data.hyperdata import (
    HyperDataImageDataset,
    HyperDataSegmentationDataset,
    WeightManager,
)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
REMOTE_S3_URL = "s3://hyperdata-data/lumen/microscopy-demo.zarr"
DATA_DIR = "/tmp/lumen_hyperdata_demo"
WEIGHTS_DIR = "/tmp/lumen_hyperdata_weights"

print(f"Device: {DEVICE}")
print(f"PyTorch: {torch.__version__}")
print(f"Endpoint: {os.environ['HYPERDATA_ENDPOINT']}")

## 1. Pull Dataset & Load with Lumen Adapters

`HyperDataImageDataset` and `HyperDataSegmentationDataset` wrap a HyperData
Zarr store into PyTorch `Dataset` objects. They handle channel conversion,
resizing, and dtype normalization so the data is ready for Lumen's trainers.

In [ ]:
# Pull the shared microscopy demo from remote S3 (IceChunk delta sync)
ds = HyperData(DATA_DIR)
ds.add_remote("origin", REMOTE_S3_URL)
result = ds.pull("origin")
ds = HyperData(DATA_DIR)
print(f"Pulled: {ds['images'].shape[0]} images, keys={ds.keys()}")

# Self-supervised loader (images only)
img_dataset = HyperDataImageDataset(ds, array_name="images", channels=1, image_size=64)
# Segmentation loader (image + mask pairs)
seg_dataset = HyperDataSegmentationDataset(
    ds, image_array="images", mask_array="masks", image_size=64, channels=1,
)

sample = seg_dataset[0]
print(f"Image: {sample['image'].shape}, Mask: {sample['mask'].shape}")
print(f"Classes in mask: {sample['mask'].unique().tolist()}")

## 2. Lumen Augmentation Pipeline

`default_seg_aug()` returns a YOLO-style transform pipeline tuned for
sim-to-real microscopy transfer. Geometric transforms (flip, rotate, affine)
are synced across image and mask; photometric transforms (brightness, gamma,
noise) only touch the image.

All transforms are pure PyTorch — no albumentations dependency.

In [ ]:
aug = default_seg_aug()

# Apply augmentation to a segmentation sample
sample = seg_dataset[0]
augmented = aug({"image": sample["image"], "label": sample["mask"]})
print(f"Original  — image: {sample['image'].shape}, mask: {sample['mask'].shape}")
print(f"Augmented — image: {augmented['image'].shape}, label: {augmented['label'].shape}")

# The pipeline includes: RandomAffine, RandomFlip, RandomRotate90,
# RandomBrightnessContrast, RandomGamma, RandomBlur, GaussianNoise, PoissonNoise
print(f"\nTransforms in pipeline: {len(aug.transforms)}")
for t in aug.transforms:
    print(f"  {type(t).__name__}(p={t.p})")

## 3. Self-Supervised Pretraining with MAETrainer

Lumen's `MAETrainer` implements Masked Autoencoder pretraining: it randomly
masks 75% of image patches and trains the encoder to reconstruct them.

Any encoder conforming to `EncoderProtocol` (attributes: `embed_dim`,
`patch_size`, `in_channels`, `supports_masked_tokens`) works with all Lumen
trainers. Below we build a tiny ViT encoder using `EncoderBase`.

> **Production note:** replace `MicroscopyEncoder` with `EUPEEncoder`
> (vendor weights at `vendor/EUPE/`) or `DINOv3Encoder` (Transformers
> checkpoint). The rest of the pipeline stays identical.

In [ ]:
class MicroscopyEncoder(EncoderBase):
    """Tiny ViT encoder for microscopy — conforms to Lumen's EncoderProtocol.

    In production, swap this for EUPEEncoder (with vendor weights) or
    DINOv3Encoder (with Transformers checkpoint). The rest of the pipeline
    stays identical because all Lumen trainers/heads accept EncoderProtocol.
    """

    def __init__(self, patch_size=16, in_channels=1, embed_dim=64, img_size=64):
        super().__init__()
        self.patch_size = patch_size
        self.in_channels = in_channels
        self.embed_dim = embed_dim
        self.supports_masked_tokens = False

        self.proj = torch.nn.Conv2d(
            in_channels, embed_dim, kernel_size=patch_size, stride=patch_size,
        )
        self.blocks = torch.nn.TransformerEncoder(
            torch.nn.TransformerEncoderLayer(
                d_model=embed_dim, nhead=4, dim_feedforward=256,
                batch_first=True, norm_first=True,
            ),
            num_layers=3,
        )
        self.norm = torch.nn.LayerNorm(embed_dim)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.proj(x)                        # (B, D, H/P, W/P)
        x = x.flatten(2).transpose(1, 2)        # (B, N, D)
        x = self.blocks(x)
        return self.norm(x)                      # (B, N, D) — patch tokens


encoder = MicroscopyEncoder(
    patch_size=16, in_channels=1, embed_dim=64, img_size=64,
)
grid = encoder.token_grid((64, 64))
print(f"Encoder: embed_dim={encoder.embed_dim}, patch_grid={grid}, "
      f"params={sum(p.numel() for p in encoder.parameters()):,}")

In [ ]:
# MAETrainer: self-supervised masked reconstruction
mae_trainer = MAETrainer(
    encoder,
    mask_ratio=0.75,
    decoder_embed_dim=64,
    decoder_depth=2,
    decoder_num_heads=4,
)
mae_optimizer = torch.optim.AdamW(mae_trainer.parameters(), lr=1e-3, weight_decay=1e-4)

# Use Lumen's train_self_supervised_epoch workflow
ssl_loader = torch.utils.data.DataLoader(img_dataset, batch_size=16, shuffle=True)

print("MAE Self-Supervised Pretraining:")
for epoch in range(5):
    metrics = train_self_supervised_epoch(
        mae_trainer, ssl_loader, mae_optimizer, device=DEVICE,
    )
    print(f"  Epoch {epoch+1}: mae_loss={metrics['mae_loss']:.4f}")

print("\nEncoder pretrained via masked reconstruction.")

## 4. Visualize Learned Features

`tokens_to_pca_rgb` projects the encoder's high-dimensional patch tokens
into an RGB image using the first 3 PCA components — a quick way to verify
the encoder has learned spatially meaningful representations.

In [ ]:
encoder.eval()
with torch.no_grad():
    sample_img = img_dataset[0]["image"].unsqueeze(0).to(DEVICE)  # (1, 1, 64, 64)
    tokens = encoder(sample_img)                                   # (1, 16, 64)

grid_h, grid_w = encoder.token_grid(sample_img.shape[2:])
pca_rgb = tokens_to_pca_rgb(
    tokens, grid_size=(grid_h, grid_w), image_size=(64, 64),
)
print(f"PCA feature map: {pca_rgb.shape}")  # (3, 64, 64)
print(f"Value range: [{pca_rgb.min():.3f}, {pca_rgb.max():.3f}]")
print("Spatially coherent PCA map indicates the encoder learned local structure.")
encoder.train();

## 5. Segmentation Fine-Tuning with SegmentationTrainer

`SegmentationTrainer` wires the pretrained encoder to a `SegmentationHead`
and handles optimizer construction, LR scheduling, loss computation, and
mixed-precision support internally.

`train_fine_tune_epoch` is Lumen's epoch loop for downstream trainers
(trainers that own their optimizer). The augmentation pipeline is applied
per-sample inside a custom dataset wrapper.

In [ ]:
NUM_CLASSES = 4  # background + 3 tissue types

seg_trainer = SegmentationTrainer(
    encoder,
    num_classes=NUM_CLASSES,
    lr=5e-4,
    weight_decay=1e-4,
    scheduler_name="cosine",
    scheduler_t_max=10,
    segmentation_loss="ce_dice",       # combined cross-entropy + soft Dice
    segmentation_ce_weight=1.0,
    segmentation_dice_weight=0.5,
    trainability="encoder_and_head",   # fine-tune both encoder and head
)
print(f"SegmentationTrainer: {NUM_CLASSES} classes, loss=ce_dice")
print(f"  Head: {type(seg_trainer.head).__name__}")
print(f"  Optimizer: {type(seg_trainer.optimizer).__name__}")

In [ ]:
# Augmented segmentation DataLoader
aug = default_seg_aug()

class AugmentedSegDataset(torch.utils.data.Dataset):
    """Wraps HyperDataSegmentationDataset with Lumen augmentations."""
    def __init__(self, base_dataset, transform):
        self.base = base_dataset
        self.transform = transform

    def __len__(self):
        return len(self.base)

    def __getitem__(self, idx):
        sample = self.base[idx]
        augmented = self.transform({"image": sample["image"], "label": sample["mask"]})
        return {"image": augmented["image"], "mask": augmented["label"]}

aug_seg_dataset = AugmentedSegDataset(seg_dataset, aug)
seg_loader = torch.utils.data.DataLoader(aug_seg_dataset, batch_size=16, shuffle=True)

# Fine-tune using Lumen's epoch workflow
print("Segmentation Fine-Tuning (with augmentation):")
for epoch in range(10):
    metrics = train_fine_tune_epoch(seg_trainer, seg_loader, device=DEVICE)
    if (epoch + 1) % 2 == 0:
        print(f"  Epoch {epoch+1}: loss={metrics['loss']:.4f}")

## 6. Evaluate with Lumen Metrics

Lumen provides microscopy-specific evaluation: `mean_iou`, `dice_coefficient`,
and `pixel_accuracy`. These work on class-index tensors and support
`ignore_index` for out-of-bounds pixels from augmentation.

In [ ]:
seg_trainer.eval()
eval_loader = torch.utils.data.DataLoader(seg_dataset, batch_size=16)

all_preds, all_targets = [], []
with torch.no_grad():
    for batch in eval_loader:
        images = batch["image"].to(DEVICE)
        masks = batch["mask"].to(DEVICE)
        logits = seg_trainer(images)                # (B, C, H, W)
        preds = logits.argmax(dim=1)                # (B, H, W)
        all_preds.append(preds.cpu())
        all_targets.append(masks.cpu())

preds = torch.cat(all_preds)
targets = torch.cat(all_targets)

miou = mean_iou(preds, targets, num_classes=NUM_CLASSES)
dice = dice_coefficient(preds, targets, num_classes=NUM_CLASSES)
px_acc = pixel_accuracy(preds, targets)

print(f"Segmentation Evaluation ({len(seg_dataset)} samples):")
print(f"  Mean IoU:         {miou:.4f}")
print(f"  Dice Coefficient: {dice:.4f}")
print(f"  Pixel Accuracy:   {px_acc:.4f}")

## 7. Push Dataset & Weights to HyperData

- `push_to_hub()` syncs Zarr data to S3 and registers the dataset in the Hub
  catalogue (visible in `hd overview`).
- `WeightManager` stores model weights with versioning, tagging, and metrics
  in a HyperData Zarr store.

In [ ]:
# Push dataset to Hub (S3 delta sync + catalogue registration)
ds.push_to_hub("@devin/microscopy-demo")
print("Dataset pushed to Hub — visible in `hd overview`")

# Push trained weights
wm = WeightManager(WEIGHTS_DIR)
wm.push_weights(
    seg_trainer,
    message=f"SegmentationTrainer: mIoU={miou:.4f}, dice={dice:.4f}",
    tag="seg-v1",
    metrics={"miou": miou, "dice": dice, "pixel_accuracy": px_acc},
)
wm.push_checkpoint(
    seg_trainer, seg_trainer.optimizer, epoch=10,
    message="Full checkpoint after 10 epochs",
)
print(f"Weight tags: {wm.list_tags()}")

## 8. Pull Weights & Resume Training

`WeightManager.pull_weights` restores model state from a tagged version.
`pull_checkpoint` also restores optimizer state for seamless training
resumption.

In [ ]:
# Build a fresh model with the same architecture
fresh_encoder = MicroscopyEncoder(patch_size=16, in_channels=1, embed_dim=64, img_size=64)
fresh_seg = SegmentationTrainer(
    fresh_encoder, num_classes=NUM_CLASSES, lr=5e-4, scheduler_name="none",
)

# Pull tagged weights
wm2 = WeightManager(WEIGHTS_DIR)
meta = wm2.pull_weights(fresh_seg, tag="seg-v1")
print(f"Restored weights — metrics: {meta.get('metrics', {})}")

# Verify the restored model produces the same predictions
fresh_seg.eval()
with torch.no_grad():
    test_img = seg_dataset[0]["image"].unsqueeze(0)
    original_pred = seg_trainer(test_img.to(DEVICE)).argmax(dim=1).cpu()
    restored_pred = fresh_seg(test_img.to(DEVICE)).argmax(dim=1).cpu()
match = (original_pred == restored_pred).float().mean()
print(f"Prediction match: {match:.1%}")

## Cleanup

In [ ]:
import shutil
shutil.rmtree(DATA_DIR, ignore_errors=True)
shutil.rmtree(WEIGHTS_DIR, ignore_errors=True)
print("Cleaned up temp directories.")